In [ ]:
# Databricks notebook source
# Kaapi Bricks MAS Evaluation
# Experiment: mas-3c936239-dev-experiment (ID: 982411422225143)

In [ ]:
%pip install -U 'mlflow[databricks]>=3.1'
%restart_python

In [ ]:
import re
import requests
import mlflow
from mlflow.genai.scorers import Correctness, Safety, RelevanceToQuery
from databricks.sdk import WorkspaceClient

MAS_ENDPOINT_NAME = "mas-3c936239-endpoint"

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(experiment_id="982411422225143")

In [ ]:
@mlflow.trace
def predict(query: str) -> str:
    w = WorkspaceClient()
    url = f"{w.config.host.rstrip('/')}/serving-endpoints/{MAS_ENDPOINT_NAME}/invocations"
    headers = w.config.authenticate()
    headers["Content-Type"] = "application/json"
    resp = requests.post(url, headers=headers, json={"input": [{"role": "user", "content": query}]}, timeout=300)
    resp.raise_for_status()
    data = resp.json()
    parts = []
    for item in data.get("output", []):
        if item.get("type") == "message":
            for block in item.get("content", []):
                if block.get("type") == "output_text":
                    parts.append(block["text"])
    full_text = "\n\n".join(parts) if parts else "No response from agent."

    # Extract the Ops_bot (KA) section and return it clean.
    ops_match = re.search(r'<name>Ops_bot</name>\s*(.*?)(?=\n<name>|\Z)', full_text, re.DOTALL)
    ops_text = ops_match.group(1).strip() if ops_match else full_text
    ops_text = re.sub(r'\n\[\^[^\]]+\]:.*?(?=\n\[\^|\n\n|\Z)', '', ops_text, flags=re.DOTALL).strip()
    ops_text = re.sub(r'\[\^[^\]]+\]', '', ops_text).strip()
    return ops_text if ops_text else "No response from agent."

In [ ]:
eval_dataset = [
    # MAS → Ops Bot (KA) — recipes
    {
        "inputs": {"query": "How do I make a Classic Filter Coffee?"},
        "expectations": {
            "expectations_guidelines": (
                "Classic Filter Coffee (PRD-001): 30ml freshly dripped decoction (house blend: 70% Chikmagalur Robusta + 30% Chicory, "
                "brewed at 92-96 degrees C, dripped 12-15 minutes), 120ml hot full cream milk, 10g sugar. "
                "Serve in traditional tumbler-davara with signature pour technique (3-4 pours for froth)."
            ),
            "expected_response": (
                "Classic Filter Coffee (PRD-001) uses 30ml freshly dripped decoction from the Kaapi Bricks house blend "
                "(70% Chikmagalur Robusta + 30% Chicory), brewed at 92-96 degrees C and dripped for 12-15 minutes. "
                "Add 120ml hot full cream milk and 10g sugar. Serve in a traditional tumbler-davara using the signature "
                "pour technique with 3-4 pours to create a rich frothy top layer."
            ),
        },
    },
    {
        "inputs": {"query": "What is the correct decoction preparation process and timing?"},
        "expectations": {
            "expectations_guidelines": (
                "Use the South Indian brass filter. Add 2 heaped tablespoons (approx 20g) of house blend per 150ml boiling water "
                "(92-96 degrees C). Allow to drip for 12-15 minutes. Maximum hold time: 4 hours before discarding."
            ),
            "expected_response": (
                "Prepare decoction using the South Indian brass filter: add 2 heaped tablespoons (approximately 20g) of "
                "Kaapi Bricks house blend per 150ml of boiling water at 92-96 degrees C. Allow the decoction to drip for "
                "12-15 minutes for proper extraction. Maximum hold time is 4 hours; discard after that as the coffee oxidizes."
            ),
        },
    },
    {
        "inputs": {"query": "How do we prepare the decoction for Classic Filter Coffee — what's the ratio?"},
        "expectations": {
            "expectations_guidelines": (
                "Decoction ratio: 2 heaped tablespoons (20g) of coffee per 150ml water at 92-96 degrees C. "
                "Use the Robusta-Chicory house blend (70:30 ratio). Drip for 12-15 minutes."
            ),
            "expected_response": (
                "For Classic Filter Coffee decoction, use 2 heaped tablespoons (20g) of coffee per 150ml of water "
                "heated to 92-96 degrees C. The house blend is 70% Chikmagalur Robusta and 30% Chicory. "
                "Allow decoction to drip for 12-15 minutes."
            ),
        },
    },

    # MAS → Ops Bot (KA) — food safety & suppliers
    {
        "inputs": {"query": "What are the allergen handling procedures for nut-based milk alternatives?"},
        "expectations": {
            "expectations_guidelines": (
                "Almond milk and coconut milk may contain tree nut allergens — warn customers with nut allergies. "
                "Use separate equipment for allergen-containing items. The blender must be cleaned between orders "
                "when switching between nut-based and dairy milks. When a customer reports an allergy: confirm the "
                "specific allergen, use freshly cleaned equipment, and inform the customer of cross-contact risks."
            ),
            "expected_response": (
                "Almond milk and coconut milk may contain tree nut allergens — customers with nut allergies must be warned. "
                "Use separate equipment for each milk type to prevent cross-contamination. The blender must be cleaned "
                "between orders when switching between nut-based and dairy milks. When a customer reports an allergy: "
                "confirm the specific allergen, check all ingredients, use freshly cleaned equipment, and inform the "
                "customer of cross-contact risks."
            ),
        },
    },
    {
        "inputs": {"query": "How long does Coorg Arabica take to reorder from our supplier?"},
        "expectations": {
            "expectations_guidelines": (
                "Coorg Arabica Beans from Coorg Coffee Estates (SUP-001): 7-day lead time. "
                "Minimum order is 50 kg. Reorder threshold is 10 kg. Typical weekly usage per store: 5-8 kg."
            ),
            "expected_response": (
                "Coorg Arabica Beans from Coorg Coffee Estates (SUP-001) has a 7-day lead time. "
                "Minimum order is 50 kg. The reorder threshold is 10 kg, and typical weekly usage per store is 5-8 kg."
            ),
        },
    },
]

In [ ]:
results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict,
    scorers=[
        Safety(),
        RelevanceToQuery(),
        Correctness(),
    ],
)

display(results.tables["eval_results"])